
# EvRare2LIM — Fixed Multilevel Splitting with a combined score

We estimate

$$
\mathbb{P}\left(N^{+2}_{\tau} \leq 10 \mid \tau = \tau_{+1}\right),
$$

using the model of Question 1.2.5.1:

- the first limits $N^{+1}$ and $N^{-1}$ follow coupled Hawkes-type dynamics;
- the second limit $N^{+2}$ follows a constant-rate birth-death process;
- the rare event is that $N^{+2}$ remains small when $N^{+1}$ is consumed first.

The splitting score favours both:

1. progress of $N^{+1}$ toward zero;
2. low values of $N^{+2}$ near the final levels.


In [ ]:

import numpy as np
import pandas as pd

from helpers.helpers_EvRare2LIM import *



## 1. Model parameters

We keep the same first-limit parameters used in the QRNIID / QR2LIM setting.

For the second limit, we use:

$$
N^{+2}_0 = 10, \qquad \lambda^{+2,+} = 1.5, \qquad \lambda^{+2,-} = 0.5.
$$

The target threshold is

$$
h = 10.
$$


In [2]:

params = ModelParams2LIM(
    q1_0=10,
    q2_0=10,
    mu_plus=1.0,
    mu_minus=1.1,
    alpha=0.50,
    beta=0.50,
    gamma_cross=1.0,
    lambda_second_add=1.500,
    lambda_second_remove=0.500,
    h=10,
    max_events=2_000_000,
)

params


ModelParams2LIM(q1_0=10, q2_0=10, mu_plus=1.0, mu_minus=1.1, alpha=0.5, beta=0.5, gamma_cross=1.0, lambda_floor=1e-08, lambda_second_add=1.5, lambda_second_remove=0.5, h=10, max_events=2000000)


## 2. Naive Monte Carlo benchmark

Before using rare-event methods, we compute a naive conditional Monte Carlo estimate. This is useful as a benchmark.

The naive estimator is

$$
\widehat p_{MC}
=
\frac{\#\{N^{+2}_{\tau}\leq h,\, \tau=\tau_{+1}\}}
{\#\{\tau=\tau_{+1}\}}.
$$


In [3]:

naive_summary = naive_monte_carlo_conditional(
    params=params,
    n_samples=100_000,
    seed=12345,
)

pd.DataFrame([naive_summary])


,method,n_samples,conditioned_samples,rare_events,p_tau_plus1,estimate,standard_error,ci95_low,ci95_high,relative_error,mean_q2_cond,mean_tau_cond
0,Naive conditional Monte Carlo,100000,49890,33,0.4989,0.000661,0.000115,0.000436,0.000887,0.17402,70.724935,60.672913



## 3. Improved splitting score

We use the score

$$
S(x)
=
(q^{+1}_0 - q^{+1})
+
 w_{2}\,\max\left(0, h + m_2 - q^{+2}\right),
$$

where:

- $q^{+1}_0 - q^{+1}$ favours trajectories where $N^{+1}$ approaches zero;
- $\max(0, h + m_2 - q^{+2})$ favours trajectories where $N^{+2}$ remains small;
- $w_2$ controls the relative importance of the second-limit term.


In [4]:

settings = FMSSettings2LIM(
    n_particles=2_000,
    n_repeats=20,
    score_levels=(2.0, 4.0, 6.0, 8.0, 10.0, 11.0, 12.0, 13.0, 14.0),
    q2_margin=20,
    w_q2=0.25,
    seed=12345,
)

settings


FMSSettings2LIM(n_particles=2000, n_repeats=20, score_levels=(2.0, 4.0, 6.0, 8.0, 10.0, 11.0, 12.0, 13.0, 14.0), q2_margin=20, w_q2=0.25, seed=12345)


## 4. One Fixed Multilevel Splitting run

This estimates the numerator

$$
\mathbb{P}\left(N^{+2}_{\tau}\leq h,\ \tau=\tau_{+1}\right).
$$

The denominator $\mathbb{P}(\tau=\tau_{+1})$ is not rare, so we estimate it separately by naive Monte Carlo.


In [5]:

one_fms = fixed_multilevel_splitting_combined_score(
    params=params,
    settings=settings,
    seed=12345,
)

print("Method:", one_fms["method"])
print("Numerator estimate:", f"{one_fms['estimate']:.6e}")
print("Failed:", one_fms["failed"])
print("Conditional probabilities:")
print(one_fms["conditional_probabilities"])
print("Survivors by level:")
print(one_fms["survivors_by_level"])


Method: FMS combined score numerator
Numerator estimate: 2.895178e-04
Failed: False
Conditional probabilities:
[1.0, 1.0, 0.8365, 0.737, 0.807, 0.1435, 0.4595, 0.3295, 0.216, 0.124]
Survivors by level:
[2000, 2000, 1673, 1474, 1614, 287, 919, 659, 432, 248]


In [6]:

level_diagnostics = pd.DataFrame(one_fms["level_infos"])
level_diagnostics


,score_level,n_survivors,p_cond,rare_absorbed,mean_q_plus1,mean_q_plus2,min_q_plus2,max_q_plus2,mean_tau,mean_score,reasons
0,2.0,2000,1.0000,0,10.000000,10.000000,10.0,10.0,0.000000,5.000000,{'reached_score_level': 2000}
1,4.0,2000,1.0000,0,10.000000,10.000000,10.0,10.0,0.000000,5.000000,{'reached_score_level': 2000}
2,6.0,1673,0.8365,0,7.813509,19.237896,6.0,453.0,9.372892,6.136133,"{'reached_score_level': 1673, 'minus1_hit_zero..."
3,8.0,1474,0.7370,0,3.764586,41.803256,6.0,299.0,32.031407,8.112958,"{'reached_score_level': 1474, 'minus1_hit_zero..."
4,10.0,1614,0.8070,0,0.723048,61.193309,5.0,313.0,51.076594,10.065830,"{'reached_score_level': 1614, 'minus1_hit_zero..."
5,11.0,287,0.1435,0,1.860627,17.585366,6.0,26.0,12.429130,11.243031,"{'terminal_plus1_but_q2_too_large': 1675, 'rea..."
6,12.0,919,0.4595,0,1.152339,16.441785,5.0,22.0,12.782347,12.237214,"{'terminal_plus1_but_q2_too_large': 946, 'reac..."
7,13.0,659,0.3295,0,0.678300,14.408194,4.0,18.0,12.734409,13.219651,"{'terminal_plus1_but_q2_too_large': 1258, 'rea..."
8,14.0,432,0.2160,0,0.368056,11.699074,5.0,14.0,12.298494,14.207176,"{'terminal_plus1_but_q2_too_large': 1515, 'rea..."
9,final_event,248,0.1240,248,0.239500,15.686500,3.0,328.0,16.267404,13.823000,"{'plus1_hit_zero': 1964, 'minus1_hit_zero': 36}"



## 5. Repeated FMS runs

We repeat the FMS estimator to estimate its variance.


In [7]:

fms_summary, fms_repeats = repeat_fms_combined_score(
    params=params,
    settings=settings,
)

pd.DataFrame([fms_summary])


,method,n_repeats,n_particles,score_levels,q2_margin,w_q2,numerator_estimate,standard_error,ci95_low,ci95_high,relative_error,n_valid_repeats,n_failed_repeats
0,Repeated FMS combined score numerator,20,2000,"(2.0, 4.0, 6.0, 8.0, 10.0, 11.0, 12.0, 13.0, 1...",20,0.25,0.000326,0.000024,0.000279,0.000373,0.073639,20,0


In [8]:

fms_repeats


,repeat,numerator_estimate,failed,failed_at_score_level
0,0,0.000281,False,None
1,1,0.000386,False,None
2,2,0.000144,False,None
3,3,0.000283,False,None
4,4,0.000407,False,None
5,5,0.000210,False,None
6,6,0.000266,False,None
7,7,0.000450,False,None
8,8,0.000326,False,None
9,9,0.000292,False,None



## 6. Denominator estimate

The denominator is

$$
\mathbb{P}(\tau=\tau_{+1}).
$$

In the symmetric first-limit model, this probability should be close to $1/2$. Since it is not rare, a naive Monte Carlo estimator is sufficient.


In [9]:

denominator_summary = estimate_denominator_tau_plus1(
    params=params,
    n_samples=100_000,
    seed=54321,
)

pd.DataFrame([denominator_summary])


,method,n_samples,tau_plus1_count,estimate,standard_error,ci95_low,ci95_high,relative_error
0,Naive Monte Carlo denominator,100000,49910,0.4991,0.001581,0.496001,0.502199,0.003168



## 7. Conditional probability estimate

We combine:

$$
\widehat p_{num}
\approx
\mathbb{P}\left(N^{+2}_{\tau}\leq h,\ \tau=\tau_{+1}\right),
$$

and

$$
\widehat p_{den}
\approx
\mathbb{P}(\tau=\tau_{+1}),
$$

using

$$
\widehat p
=
\frac{\widehat p_{num}}{\widehat p_{den}}.
$$


In [10]:

conditional_summary = conditional_from_numerator_and_denominator(
    numerator_summary=fms_summary,
    denominator_summary=denominator_summary,
)

pd.DataFrame([conditional_summary])


,method,numerator_estimate,denominator_estimate,estimate,standard_error,ci95_low,ci95_high,relative_error
0,FMS numerator / MC denominator,0.000326,0.4991,0.000653,0.000048,0.000559,0.000747,0.073707


In [11]:

comparison = pd.DataFrame([
    {
        "method": "Naive conditional MC",
        "estimate": naive_summary["estimate"],
        "standard_error": naive_summary["standard_error"],
        "ci95_low": naive_summary["ci95_low"],
        "ci95_high": naive_summary["ci95_high"],
        "relative_error": naive_summary["relative_error"],
    },
    {
        "method": "FMS combined score",
        "estimate": conditional_summary["estimate"],
        "standard_error": conditional_summary["standard_error"],
        "ci95_low": conditional_summary["ci95_low"],
        "ci95_high": conditional_summary["ci95_high"],
        "relative_error": conditional_summary["relative_error"],
    },
])

comparison


,method,estimate,standard_error,ci95_low,ci95_high,relative_error
0,Naive conditional MC,0.000661,0.000115,0.000436,0.000887,0.174020
1,FMS combined score,0.000653,0.000048,0.000559,0.000747,0.073707
